In [43]:
import gdsfactory as gf
from gdsfactory.generic_tech import get_generic_pdk


PDK = get_generic_pdk()
PDK.activate()


layer_views = PDK.layer_views
print(layer_views)

d:\conda\envs\gdsenv\Lib\site-packages\gdsfactory\gpdk\layer_views.yaml


In [56]:
import gdsfactory as gf

g = 0.2 # coupling gap
Lc = 18 # coupling length straight wg
Dx = 20 # bend in x 
Dy = 10 # bend in y

coupler = gf.components.coupler(
    gap=g,
    length=Lc,
    dy=Dy,
    dx=Dx,
    cross_section="strip",  
)

# floor plan 
floorplan = gf.Component()
coupler_ref = floorplan.add_ref(coupler)
# coupler_ref.move((0, 0))

# size of floor
Fx = 3000
Fy = 8000
floorplan.add_polygon(
    [(-Fx/2, -Fy/2), (Fx/2, -Fy/2), (Fx/2, Fy/2), (-Fx/2, Fy/2)],
    layer=(100, 0),
)

# get four ports
port_o1 = coupler_ref.ports["o1"]
port_o2 = coupler_ref.ports["o2"]
port_o3 = coupler_ref.ports["o3"]
port_o4 = coupler_ref.ports["o4"]


length_to_left_edge = Fx/2 - abs(port_o1.center[0])
length_to_right_edge = Fx/2 - abs(port_o3.center[0])

wg_o1 = gf.components.straight(length=length_to_left_edge, cross_section='strip')
wg_o1_ref = floorplan.add_ref(wg_o1)
wg_o1_ref.connect("o2", port_o1)

wg_o2 = gf.components.straight(length=length_to_left_edge, cross_section='strip')
wg_o2_ref = floorplan.add_ref(wg_o2)
wg_o2_ref.connect("o2", port_o2)

wg_o3 = gf.components.straight(length=length_to_right_edge, cross_section='strip')
wg_o3_ref = floorplan.add_ref(wg_o3)
wg_o3_ref.connect("o1", port_o3)

wg_o4 = gf.components.straight(length=length_to_right_edge, cross_section='strip')
wg_o4_ref = floorplan.add_ref(wg_o4)
wg_o4_ref.connect("o1", port_o4)

#### comparision straight wg
wg_comp = gf.components.straight(length = Fx, cross_section= 'strip')
wg_comp_ref = floorplan.add_ref(wg_comp)

wg_comp_ref.center = (0,500)             
wg_comp_ref.rotation = 0                   

### add the slab layer

# define slab size
ox = 20
oy = 2
slab_x = Lc + 2*Dx + ox
slab_y = 2*Dy + oy


floorplan.add_polygon(
    [(-slab_x/2 + Lc/2, -slab_y/2), (slab_x/2 + Lc/2, -slab_y/2), \
     (slab_x/2 + Lc/2, slab_y/2), (-slab_x/2 + Lc/2, slab_y/2)],
    layer=(3, 0),
)

floorplan.show()
floorplan.write_gds("directional_coupler_floorplan.gds")

WindowsPath('directional_coupler_floorplan.gds')

In [65]:
import gdsfactory as gf
import numpy as np
from gdsfactory.generic_tech import get_generic_pdk

### generating a series of 3 db couplers with diff gaps

gaps = np.linspace(0.15, 0.25, 5)
Lc = 18 # coupling length straight wg
Dx = 20 # bend in x 
Dy = 10 # bend in y
cross_section = "strip"

chip = gf.Component()
# size of chip
Fx = 3000
Fy = 8000
chip.add_polygon(
    [(-Fx/2, -Fy/2), (Fx/2, -Fy/2), (Fx/2, Fy/2), (-Fx/2, Fy/2)],
    layer=(100, 0),
)


## device spacing in y direction
spacing_y = 100.0   

i = 0
for gap in gaps:

    coupler = gf.components.coupler(
        gap= gap,
        length= Lc,
        dy= Dy,
        dx = Dx,
        cross_section=cross_section,
    )

    coupler_ref = chip.add_ref(coupler)
    coupler_ref.movey(i * spacing_y)

    # get four ports
    port_o1 = coupler_ref.ports["o1"]
    port_o2 = coupler_ref.ports["o2"]
    port_o3 = coupler_ref.ports["o3"]
    port_o4 = coupler_ref.ports["o4"]


    length_to_left_edge = Fx/2 - abs(port_o1.center[0])
    length_to_right_edge = Fx/2 - abs(port_o3.center[0])

    wg_o1 = gf.components.straight(length=length_to_left_edge, cross_section='strip')
    wg_o1_ref = chip.add_ref(wg_o1)
    wg_o1_ref.connect("o2", port_o1)

    wg_o2 = gf.components.straight(length=length_to_left_edge, cross_section='strip')
    wg_o2_ref = chip.add_ref(wg_o2)
    wg_o2_ref.connect("o2", port_o2)

    wg_o3 = gf.components.straight(length=length_to_right_edge, cross_section='strip')
    wg_o3_ref = chip.add_ref(wg_o3)
    wg_o3_ref.connect("o1", port_o3)

    wg_o4 = gf.components.straight(length=length_to_right_edge, cross_section='strip')
    wg_o4_ref = chip.add_ref(wg_o4)
    wg_o4_ref.connect("o1", port_o4)

    # define slab size
    ox = 20
    oy = 2
    slab_x = Lc + 2*Dx + ox
    slab_y = 2*Dy + oy


    chip.add_polygon(
        [(-slab_x/2 + Lc/2, -slab_y/2 + i*spacing_y), (slab_x/2 + Lc/2, -slab_y/2+ i*spacing_y), \
        (slab_x/2 + Lc/2, slab_y/2+ i*spacing_y), (-slab_x/2 + Lc/2, slab_y/2+ i*spacing_y)],
        layer=(3, 0),
    )

    label = gf.components.text(
        text=f"gap={gap}",
        size=3.0,           
        layer=(10, 0),      
    )
    label_ref = chip.add_ref(label)
    label_ref.movey(
        (i * spacing_y - 15)  
    )
    i = i + 1

chip.show()

chip.write_gds("coupler_gap_sweep.gds")

WindowsPath('coupler_gap_sweep.gds')